In [11]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Customer_support_chatbot'
!pip install -q datasets scikit-learn

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


load dataset

In [12]:
from datasets import load_dataset

lang_ds = load_dataset("papluca/language-identification")
print(lang_ds)

train_df = lang_ds['train'].to_pandas()
val_df = lang_ds['validation'].to_pandas()
test_df = lang_ds['test'].to_pandas()

train_df.head

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})


<bound method NDFrame.head of       labels                                               text
0         pt  os chefes de defesa da estónia, letónia, lituâ...
1         bg  размерът на хоризонталната мрежа може да бъде ...
2         zh  很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把...
3         th  สำหรับ ของเก่า ที่ จริงจัง ลอง   honeychurch  ...
4         ru                             Он увеличил давление .
...      ...                                                ...
69995     ja  本格的なゲーミングヘッドホンでした。 今まで使ってた1万円するパナソニックのヘッドホンは何だ...
69996     el  Ναι , ξέρω ένα που είναι ακόμα έτσι , αλλά αυτ...
69997     ur  اور مجھے اس ملک کے بارے میں معلوم نہیں ہے کہ گ...
69998     es  Se me rompió uno al sacarlo del cargador. Cali...
69999     hi  कोसोवो कथा का विवरण जिसमें स ् थानीय राष ् ट ्...

[70000 rows x 2 columns]>

turn text into numbers (TF-IDF)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer='char_wb',
                             ngram_range=(1,3),
                             max_features=5000)

X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

y_train = train_df['labels']
y_val = val_df['labels']
y_test = test_df['labels']

Train the classifier

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

Validation accuracy: 0.9933
              precision    recall  f1-score   support

          ar       1.00      0.99      1.00       500
          bg       1.00      0.99      0.99       500
          de       1.00      0.99      1.00       500
          el       1.00      1.00      1.00       500
          en       0.99      1.00      0.99       500
          es       1.00      0.99      0.99       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.95      0.98       500
          it       0.99      1.00      0.99       500
          ja       1.00      1.00      1.00       500
          nl       0.98      1.00      0.99       500
          pl       1.00      1.00      1.00       500
          pt       0.99      0.99      0.99       500
          ru       0.99      1.00      1.00       500
          sw       0.94      1.00      0.97       500
          th       1.00      1.00      1.00       500
          tr       1.00      1.00      1.00       500

Checking test performance, then saving all


In [15]:
test_preds = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_preds))

import joblib, os
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
joblib.dump(clf, f'{PROJECT_DIR}/models/lang_classifier.pkl')
joblib.dump(vectorizer, f'{PROJECT_DIR}/models/lang_vectorizer.pkl')

Test accuracy: 0.9928


['/content/drive/MyDrive/Customer_support_chatbot/models/lang_vectorizer.pkl']

quick sanity test

In [21]:
def detect_lang(text):
  vec = vectorizer.transform([text])
  return clf.predict(vec)[0]

print(detect_lang("Where is my order?"))               #expect en
print(detect_lang("ich bin mariam"))                   #expect es or pt
print(detect_lang("ich bin hier"))                     # no proper noun
print(detect_language_debug("ich bin ein student"))    # no proper noun

en
tr
de
de: 0.816
nl: 0.101
it: 0.016
None
